# Example queries: `restrict_avoid` (comstock_oedi_agg)

Auto-generated from `tests/query_snapshots/restrict_avoid.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
from buildstock_query.schema.utilities import MappedColumn
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_agg_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_agg_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_agg_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading comstock_amy2018_r2_2025 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `restrict_single_state`

Annual electricity restricted to CO. Three equivalent restrict shapes (single-element list, scalar string, scalar wrapped in tuple) must all compile to the same SQL — covers the arg-normalization code path in _get_restrict_clauses.


In [3]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,metadata_rows_count,model_count,units_count,electricity.total.energy_consumption..kwh
0,FullServiceRestaurant,2526,266,4556.431396,1.209202e+09
1,Hospital,63,30,152.003100,1.369473e+09
2,LargeHotel,1460,264,3317.336942,1.128185e+09
3,LargeOffice,866,184,849.927533,1.546956e+09
4,MediumOffice,2351,349,2418.404703,1.298819e+09


## `restrict_two_states`

Annual electricity restricted to CO + WY.


In [4]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['state'],
    restrict=[('state', ['CO', 'WY'])],
)
result.head() if hasattr(result, 'head') else result


,state,metadata_rows_count,model_count,units_count,electricity.total.energy_consumption..kwh
0,CO,25743,3272,84415.712040,1.600424e+10
1,WY,8469,1715,11666.235991,1.768901e+09


## `restrict_state_plus_vintage`

CO + specific vintage bucket, electricity.


In [5]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['vintage'],
    restrict=[('state', ['CO']), ('vintage', ['1980 to 1989'])],
)
result.head() if hasattr(result, 'head') else result


,vintage,metadata_rows_count,model_count,units_count,electricity.total.energy_consumption..kwh
0,1980 to 1989,4159,577,11994.992091,2.733411e+09


## `avoid_building_type`

CO only, avoid one building type per schema.


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    avoid=[('comstock_building_type', ['Warehouse'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,metadata_rows_count,model_count,units_count,electricity.total.energy_consumption..kwh
0,FullServiceRestaurant,2526,266,4556.431396,1.209202e+09
1,Hospital,63,30,152.003100,1.369473e+09
2,LargeHotel,1460,264,3317.336942,1.128185e+09
3,LargeOffice,866,184,849.927533,1.546956e+09
4,MediumOffice,2351,349,2418.404703,1.298819e+09


## `avoid_building_type_multi`

CO only, avoid multiple building types via multi-element avoid list — exercises the NOT IN code path in _add_avoid (single-value avoid emits != instead).


In [7]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption..kwh'],
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
    avoid=[('comstock_building_type', ['Warehouse', 'SmallOffice'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,metadata_rows_count,model_count,units_count,electricity.total.energy_consumption..kwh
0,FullServiceRestaurant,2526,266,4556.431396,1.209202e+09
1,Hospital,63,30,152.003100,1.369473e+09
2,LargeHotel,1460,264,3317.336942,1.128185e+09
3,LargeOffice,866,184,849.927533,1.546956e+09
4,MediumOffice,2351,349,2418.404703,1.298819e+09
